# Engine B SFT Notebook

End-to-end preparation and supervised fine-tuning workflow for Engine B.

In [1]:
import os
import json
from pathlib import Path

## 1. Paths and Configuration

Define project roots and dataset locations.

In [2]:
# Paths / config

from pathlib import Path
RUN_TRAINING = False

# Notebook location:
# gruve-ai-expo/engine_b_qwen/notebooks/engine_b_sft.ipynb

PROJECT_ROOT = Path.cwd().parent.parent

TRAIN_OCR_DIR = PROJECT_ROOT / "dataset" / "train" / "ocr"
TRAIN_GT_DIR  = PROJECT_ROOT / "dataset" / "train" / "entities"

TEST_OCR_DIR  = PROJECT_ROOT / "dataset" / "test" / "ocr"
TEST_GT_DIR   = PROJECT_ROOT / "dataset" / "test" / "entities"

OUTPUT_DIR = PROJECT_ROOT / "engine_b_qwen" / "outputs_engine_b"
NOTEBOOK_DIR = Path.cwd()

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("TRAIN_OCR_DIR exists:", TRAIN_OCR_DIR.exists(), TRAIN_OCR_DIR)
print("TRAIN_GT_DIR exists :", TRAIN_GT_DIR.exists(), TRAIN_GT_DIR)
print("TEST_OCR_DIR exists :", TEST_OCR_DIR.exists(), TEST_OCR_DIR)
print("TEST_GT_DIR exists  :", TEST_GT_DIR.exists(), TEST_GT_DIR)
print("OUTPUT_DIR exists   :", OUTPUT_DIR.exists(), OUTPUT_DIR)

PROJECT_ROOT: /home/kim.j14/gruve-ai-expo
TRAIN_OCR_DIR exists: True /home/kim.j14/gruve-ai-expo/dataset/train/ocr
TRAIN_GT_DIR exists : True /home/kim.j14/gruve-ai-expo/dataset/train/entities
TEST_OCR_DIR exists : True /home/kim.j14/gruve-ai-expo/dataset/test/ocr
TEST_GT_DIR exists  : True /home/kim.j14/gruve-ai-expo/dataset/test/entities
OUTPUT_DIR exists   : True /home/kim.j14/gruve-ai-expo/engine_b_qwen/outputs_engine_b


## 2. Dataset Sanity Checks

Verify OCR and entity files line up by document ID.

In [3]:
# sanity check - list files and match IDs
train_ocr_files = sorted(TRAIN_OCR_DIR.glob("*.txt"))
train_gt_files = sorted(TRAIN_GT_DIR.glob("*.txt"))

test_ocr_files = sorted(TEST_OCR_DIR.glob("*.txt"))
test_gt_files = sorted(TEST_GT_DIR.glob("*.txt"))

print(f"Train OCR files: {len(train_ocr_files)}")
print(f"Train GT files : {len(train_gt_files)}")
print(f"Test OCR files : {len(test_ocr_files)}")
print(f"Test GT files  : {len(test_gt_files)}")

train_ocr_ids = {f.stem for f in train_ocr_files}
train_gt_ids = {f.stem for f in train_gt_files}
test_ocr_ids = {f.stem for f in test_ocr_files}
test_gt_ids = {f.stem for f in test_gt_files}

train_common = sorted(train_ocr_ids & train_gt_ids)
test_common = sorted(test_ocr_ids & test_gt_ids)

print(f"\nMatched train pairs: {len(train_common)}")
print(f"Matched test pairs : {len(test_common)}")

print("\nFirst 10 matched train IDs:")
for doc_id in train_common[:10]:
    print("-", doc_id)

print("\nFirst 10 matched test IDs:")
for doc_id in test_common[:10]:
    print("-", doc_id)

Train OCR files: 626
Train GT files : 626
Test OCR files : 347
Test GT files  : 347

Matched train pairs: 626
Matched test pairs : 347

First 10 matched train IDs:
- X00016469612
- X00016469619
- X00016469620
- X00016469622
- X00016469623
- X00016469669
- X00016469672
- X00016469676
- X51005200938
- X51005230617

First 10 matched test IDs:
- X00016469670
- X00016469671
- X51005200931
- X51005230605
- X51005230616
- X51005230621
- X51005230648
- X51005230657
- X51005230659
- X51005268275


## 3. Single Example Inspection

Load one matched training sample to inspect raw inputs.

In [4]:
# Load one matched train example and inspect it

sample_id = train_common[0]

ocr_path = TRAIN_OCR_DIR / f"{sample_id}.txt"
gt_path = TRAIN_GT_DIR / f"{sample_id}.txt"

with open(ocr_path, "r", encoding="utf-8") as f:
    ocr_text = f.read()

with open(gt_path, "r", encoding="utf-8") as f:
    gt_text = f.read()

print("Sample ID:", sample_id)
print("\n=== OCR TEXT ===\n")
print(ocr_text[:3000])  # truncate if huge

print("\n=== GROUND TRUTH RAW ===\n")
print(gt_text[:3000])

Sample ID: X00016469612

=== OCR TEXT ===

tan woon yann

BOOK TA-K (TAMAN DAYA) SDN BHD
B94 7-W
NO.5? 55,57 & 59, JALAN SAGU 18,
TAMAN DAYA
81100 JOHOR BAHRU,
JOHOR.

LAM MCU

Document Ho : TDO1167104

 

Date 25/12/2018 8:13:39 PM
Cashier MANIS
Member
CASH BILL
CODE/DESC PRICE — Disc AMOUIT
Quy RM RM
9556939040118 KF MODELLING CLAY KIDDY FISH
1PC * 9.00) 6,00 9.00
Total : 9.00
Rour ding Adjustment 0.00

Round: :d Total (RM):

9.60

Cash
CHANGE

  

GOODS SOLD ARE NOT RETURNAR
EXCHANGEABLE

 

THANK YOU.
PLEASE COME AGATY t


=== GROUND TRUTH RAW ===

{
    "company": "BOOK TA .K (TAMAN DAYA) SDN BHD",
    "date": "25/12/2018",
    "address": "NO.53 55,57 & 59, JALAN SAGU 18, TAMAN DAYA, 81100 JOHOR BAHRU, JOHOR.",
    "total": "9.00"
}


## 4. Formatting Helpers

Build training targets and message-format utilities.

In [5]:
def load_ground_truth(gt_path: Path):
    with open(gt_path, "r", encoding="utf-8") as f:
        return json.load(f)


def build_target_json(doc_id: str, gt: dict, engine_name: str = "qwen-engine-b-sft"):
    return {
        "engine": engine_name,
        "source_file": f"{doc_id}.txt",
        "extracted_fields": {
            "company": gt.get("company"),
            "date": gt.get("date"),
            "address": gt.get("address"),
            "total": gt.get("total"),
        },
    }


gt_dict = load_ground_truth(gt_path)
target_json = build_target_json(sample_id, gt_dict)

print("=== PARSED GROUND TRUTH DICT ===\n")
print(json.dumps(gt_dict, indent=2, ensure_ascii=False))

print("\n=== FINAL TARGET JSON ===\n")
print(json.dumps(target_json, indent=2, ensure_ascii=False))

=== PARSED GROUND TRUTH DICT ===

{
  "company": "BOOK TA .K (TAMAN DAYA) SDN BHD",
  "date": "25/12/2018",
  "address": "NO.53 55,57 & 59, JALAN SAGU 18, TAMAN DAYA, 81100 JOHOR BAHRU, JOHOR.",
  "total": "9.00"
}

=== FINAL TARGET JSON ===

{
  "engine": "qwen-engine-b-sft",
  "source_file": "X00016469612.txt",
  "extracted_fields": {
    "company": "BOOK TA .K (TAMAN DAYA) SDN BHD",
    "date": "25/12/2018",
    "address": "NO.53 55,57 & 59, JALAN SAGU 18, TAMAN DAYA, 81100 JOHOR BAHRU, JOHOR.",
    "total": "9.00"
  }
}


In [6]:
SYSTEM_PROMPT = """You are an information extraction system.

Extract structured information from OCR receipt text.

Return the result as a valid JSON object with the following structure:

{
  "engine": "qwen-engine-b-sft",
  "source_file": "<filename>",
  "extracted_fields": {
    "company": null,
    "date": null,
    "address": null,
    "total": null
  }
}

Replace null with the extracted values from the receipt.

Only return valid JSON. Do not include explanations.
"""


def format_training_example(ocr_text: str, target_json: dict):

    user_prompt = f"""Extract the company, date, address, and total from the following OCR receipt text.

OCR TEXT:
{ocr_text}
"""

    assistant_output = json.dumps(target_json, indent=2, ensure_ascii=False)

    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
            {"role": "assistant", "content": assistant_output},
        ]
    }

In [7]:
example = format_training_example(ocr_text, target_json)

print(json.dumps(example, indent=2, ensure_ascii=False)[:2000])

{
  "messages": [
    {
      "role": "system",
      "content": "You are an information extraction system.\n\nExtract structured information from OCR receipt text.\n\nReturn the result as a valid JSON object with the following structure:\n\n{\n  \"engine\": \"qwen-engine-b-sft\",\n  \"source_file\": \"<filename>\",\n  \"extracted_fields\": {\n    \"company\": null,\n    \"date\": null,\n    \"address\": null,\n    \"total\": null\n  }\n}\n\nReplace null with the extracted values from the receipt.\n\nOnly return valid JSON. Do not include explanations.\n"
    },
    {
      "role": "user",
      "content": "Extract the company, date, address, and total from the following OCR receipt text.\n\nOCR TEXT:\ntan woon yann\n\nBOOK TA-K (TAMAN DAYA) SDN BHD\nB94 7-W\nNO.5? 55,57 & 59, JALAN SAGU 18,\nTAMAN DAYA\n81100 JOHOR BAHRU,\nJOHOR.\n\nLAM MCU\n\nDocument Ho : TDO1167104\n\n \n\nDate 25/12/2018 8:13:39 PM\nCashier MANIS\nMember\nCASH BILL\nCODE/DESC PRICE — Disc AMOUIT\nQuy RM RM\n955693

## 5. Build Training Examples

Convert all matched samples into chat-format training records.

In [8]:
training_examples = []

for doc_id in train_common:

    ocr_path = TRAIN_OCR_DIR / f"{doc_id}.txt"
    gt_path = TRAIN_GT_DIR / f"{doc_id}.txt"

    with open(ocr_path, "r", encoding="utf-8") as f:
        ocr_text = f.read()

    gt_dict = load_ground_truth(gt_path)

    target_json = build_target_json(doc_id, gt_dict)

    example = format_training_example(ocr_text, target_json)

    training_examples.append(example)

print("Training examples created:", len(training_examples))

Training examples created: 626


In [9]:
print(json.dumps(training_examples[0], indent=2)[:2000])

{
  "messages": [
    {
      "role": "system",
      "content": "You are an information extraction system.\n\nExtract structured information from OCR receipt text.\n\nReturn the result as a valid JSON object with the following structure:\n\n{\n  \"engine\": \"qwen-engine-b-sft\",\n  \"source_file\": \"<filename>\",\n  \"extracted_fields\": {\n    \"company\": null,\n    \"date\": null,\n    \"address\": null,\n    \"total\": null\n  }\n}\n\nReplace null with the extracted values from the receipt.\n\nOnly return valid JSON. Do not include explanations.\n"
    },
    {
      "role": "user",
      "content": "Extract the company, date, address, and total from the following OCR receipt text.\n\nOCR TEXT:\ntan woon yann\n\nBOOK TA-K (TAMAN DAYA) SDN BHD\nB94 7-W\nNO.5? 55,57 & 59, JALAN SAGU 18,\nTAMAN DAYA\n81100 JOHOR BAHRU,\nJOHOR.\n\nLAM MCU\n\nDocument Ho : TDO1167104\n\n \n\nDate 25/12/2018 8:13:39 PM\nCashier MANIS\nMember\nCASH BILL\nCODE/DESC PRICE \u2014 Disc AMOUIT\nQuy RM RM\n9

## 6. Dataset Packaging

Install and prepare Hugging Face `Dataset` objects.

In [10]:
!pip install datasets

Defaulting to user installation because normal site-packages is not writeable


In [11]:
from datasets import Dataset

train_dataset = Dataset.from_list(training_examples)

print(train_dataset)

/home/kim.j14/gruve-ai-expo/qwen_env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset({
    features: ['messages'],
    num_rows: 626
})


## 7. Training Dependencies

Install libraries required for QLoRA/SFT training.

In [12]:
!pip install unsloth datasets transformers trl peft accelerate bitsandbytes

Defaulting to user installation because normal site-packages is not writeable


**The following cells require a supported GPU environment (e.g., Northeastern Explorer cluster).
They may not run on a local Mac environment.**  
- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - 

## 8. GPU Training Setup

Load model/tooling and configure training on a supported GPU environment.

In [13]:
import torch
from datasets import Dataset
from transformers import TrainingArguments
from trl import SFTTrainer

HAS_GPU = torch.cuda.is_available()

print("CUDA available:", HAS_GPU)
if HAS_GPU:
    try:
        from unsloth import FastLanguageModel
        print("FastLanguageModel imported successfully.")
    except Exception as e:
        print("Unsloth import failed:", e)
        FastLanguageModel = None
else:
    print("Skipping Unsloth import locally; run model/training cells on HPC.")
    FastLanguageModel = None

CUDA available: True
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/tmp/ipykernel_63882/4182293727.py:11: UserWarning: WARNING: Unsloth should be imported before trl, transformers, peft to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth Zoo will now patch everything to make training faster!
FastLanguageModel imported successfully.


In [14]:
MODEL_ID = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 2048
SFT_OUTPUT_DIR = PROJECT_ROOT / "engine_b_qwen" / "outputs_engine_b" / "qwen_sft"
SFT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("MODEL_ID:", MODEL_ID)
print("MAX_SEQ_LENGTH:", MAX_SEQ_LENGTH)
print("SFT_OUTPUT_DIR:", SFT_OUTPUT_DIR)

MODEL_ID: unsloth/Qwen2.5-7B-Instruct-bnb-4bit
MAX_SEQ_LENGTH: 2048
SFT_OUTPUT_DIR: /home/kim.j14/gruve-ai-expo/engine_b_qwen/outputs_engine_b/qwen_sft


In [15]:
if FastLanguageModel is None:
    print("Model loading skipped. Run this cell on HPC with GPU support.")
else:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_ID,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
    )
    print("Model and tokenizer loaded.")

==((====))==  Unsloth 2025.11.1: Fast Qwen2 patching. Transformers: 4.57.2.
   \\   /|    Tesla V100-SXM2-32GB. Num GPUs = 1. Max memory: 31.739 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Model and tokenizer loaded.


In [16]:
if FastLanguageModel is None:
    print("LoRA setup skipped. Run this cell on HPC with GPU support.")
else:
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=[
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj",
        ],
        lora_alpha=16,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
    )
    print("LoRA adapters attached.")

Unsloth 2025.11.1 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


LoRA adapters attached.


## 9. Prompt/Data Preview

Inspect final message structure before tokenization and training.

In [17]:
print("Example message structure:\n")
print(json.dumps(training_examples[0], indent=2)[:2000])

print("\nDataset size:", len(training_examples))
print("Keys in one example:", training_examples[0].keys())
print("Keys in messages[0]:", training_examples[0]["messages"][0].keys())

Example message structure:

{
  "messages": [
    {
      "role": "system",
      "content": "You are an information extraction system.\n\nExtract structured information from OCR receipt text.\n\nReturn the result as a valid JSON object with the following structure:\n\n{\n  \"engine\": \"qwen-engine-b-sft\",\n  \"source_file\": \"<filename>\",\n  \"extracted_fields\": {\n    \"company\": null,\n    \"date\": null,\n    \"address\": null,\n    \"total\": null\n  }\n}\n\nReplace null with the extracted values from the receipt.\n\nOnly return valid JSON. Do not include explanations.\n"
    },
    {
      "role": "user",
      "content": "Extract the company, date, address, and total from the following OCR receipt text.\n\nOCR TEXT:\ntan woon yann\n\nBOOK TA-K (TAMAN DAYA) SDN BHD\nB94 7-W\nNO.5? 55,57 & 59, JALAN SAGU 18,\nTAMAN DAYA\n81100 JOHOR BAHRU,\nJOHOR.\n\nLAM MCU\n\nDocument Ho : TDO1167104\n\n \n\nDate 25/12/2018 8:13:39 PM\nCashier MANIS\nMember\nCASH BILL\nCODE/DESC PRICE \u20

In [18]:
def render_messages_as_text(example):
    parts = []
    for msg in example["messages"]:
        role = msg["role"].strip().lower()
        content = msg["content"].strip()
        parts.append(f"<|{role}|>\n{content}")
    return {"text": "\n\n".join(parts)}

train_dataset_text = train_dataset.map(render_messages_as_text)

print(train_dataset_text)
print("\n=== Sample rendered training text ===\n")
print(train_dataset_text[0]["text"][:3000])

Map: 100%|██████████| 626/626 [00:00<00:00, 15238.47 examples/s]

Dataset({
    features: ['messages', 'text'],
    num_rows: 626
})

=== Sample rendered training text ===

<|system|>
You are an information extraction system.

Extract structured information from OCR receipt text.

Return the result as a valid JSON object with the following structure:

{
  "engine": "qwen-engine-b-sft",
  "source_file": "<filename>",
  "extracted_fields": {
    "company": null,
    "date": null,
    "address": null,
    "total": null
  }
}

Replace null with the extracted values from the receipt.

Only return valid JSON. Do not include explanations.

<|user|>
Extract the company, date, address, and total from the following OCR receipt text.

OCR TEXT:
tan woon yann

BOOK TA-K (TAMAN DAYA) SDN BHD
B94 7-W
NO.5? 55,57 & 59, JALAN SAGU 18,
TAMAN DAYA
81100 JOHOR BAHRU,
JOHOR.

LAM MCU

Document Ho : TDO1167104

 

Date 25/12/2018 8:13:39 PM
Cashier MANIS
Member
CASH BILL
CODE/DESC PRICE — Disc AMOUIT
Quy RM RM
9556939040118 KF MODELLING CLAY KIDDY FISH
1PC * 9.00) 6,00 9

## 10. Tokenization Checks

Validate token counts and truncation behavior.

In [19]:
if "tokenizer" not in globals() or tokenizer is None:
    print("Tokenizer not loaded yet. Run tokenization checks on HPC after model/tokenizer load.")
else:
    sample_text = train_dataset_text[0]["text"]
    tokenized = tokenizer(sample_text, truncation=False)
    print("Sample token count:", len(tokenized["input_ids"]))

Sample token count: 528


## 11. SFT Hyperparameters and Run

Set training arguments and launch supervised fine-tuning.

In [20]:
TRAIN_BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 4
NUM_EPOCHS = 2
LEARNING_RATE = 2e-4
LOGGING_STEPS = 10
SAVE_STEPS = 100
WARMUP_STEPS = 10

print({
    "batch_size": TRAIN_BATCH_SIZE,
    "grad_accum_steps": GRAD_ACCUM_STEPS,
    "epochs": NUM_EPOCHS,
    "lr": LEARNING_RATE,
    "logging_steps": LOGGING_STEPS,
    "save_steps": SAVE_STEPS,
    "warmup_steps": WARMUP_STEPS,
})

{'batch_size': 2, 'grad_accum_steps': 4, 'epochs': 2, 'lr': 0.0002, 'logging_steps': 10, 'save_steps': 100, 'warmup_steps': 10}


In [21]:
from trl import SFTTrainer, SFTConfig

if FastLanguageModel is None or "model" not in globals() or "tokenizer" not in globals():
    print("Trainer setup skipped locally. Run on HPC after model/tokenizer load.")
else:
    training_args = SFTConfig(
        output_dir=str(SFT_OUTPUT_DIR),
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        logging_steps=LOGGING_STEPS,
        save_steps=SAVE_STEPS,
        warmup_steps=WARMUP_STEPS,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        report_to="none",
        remove_unused_columns=False,
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
    )

    trainer = SFTTrainer(
        model=model,
        processing_class=tokenizer,
        train_dataset=train_dataset_text,
        args=training_args,
    )

    print("Trainer initialized successfully.")

Unsloth: Tokenizing ["text"] (num_proc=32): 100%|██████████| 626/626 [00:06<00:00, 91.45 examples/s] 

Trainer initialized successfully.


In [22]:
if not RUN_TRAINING:
    print("Training skipped.")
else:
    trainer.train()

Training skipped.


## 12. Load Model

In [23]:
from unsloth import FastLanguageModel
from peft import PeftModel

BASE_MODEL = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
ADAPTER_DIR = "/home/kim.j14/gruve-ai-expo/engine_b_qwen/outputs_engine_b/qwen_sft/checkpoint-158"
max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

# load the trained LoRA adapter
model = PeftModel.from_pretrained(model, ADAPTER_DIR)

model.eval()

==((====))==  Unsloth 2025.11.1: Fast Qwen2 patching. Transformers: 4.57.2.
   \\   /|    Tesla V100-SXM2-32GB. Num GPUs = 1. Max memory: 31.739 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(152064, 3584, padding_idx=151654)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3584, out_features=3584, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3584, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=3584, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

## 13. Evaluation

In [24]:
FIELDS = ["company", "date", "address", "total"]

def build_inference_prompt(ocr_text: str, source_file: str):
    system = """You are an information extraction system.

Extract structured information from OCR receipt text.

Return the result as a valid JSON object with the following structure:

{
  "engine": "qwen-engine-b-sft",
  "source_file": "<filename>",
  "extracted_fields": {
    "company": null,
    "date": null,
    "address": null,
    "total": null
  }
}

Replace null with the extracted values from the receipt.
Only return valid JSON. Do not include explanations.
"""

    user = f"""Extract the company, date, address, and total from the following OCR receipt text.

OCR TEXT:
{ocr_text}
"""

    return f"<|system|>\n{system}\n\n<|user|>\n{user}\n\n<|assistant|>\n"

In [25]:
def extract_first_json_block(text: str):
    start = text.find("{")
    if start == -1:
        return None

    depth = 0
    for i in range(start, len(text)):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                return text[start:i+1]
    return None


def generate_prediction(ocr_text: str, source_file: str, max_new_tokens: int = 256):
    prompt = build_inference_prompt(ocr_text, source_file)

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
    **inputs,
    max_new_tokens=160,
    do_sample=False,
    temperature=0.0,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.eos_token_id,
    )

    # decode only newly generated tokens
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    decoded = tokenizer.decode(new_tokens, skip_special_tokens=True)

    json_text = extract_first_json_block(decoded)

    try:
        pred = json.loads(json_text) if json_text is not None else {
            "engine": "qwen-engine-b-sft",
            "source_file": source_file,
            "extracted_fields": {
                "company": None,
                "date": None,
                "address": None,
                "total": None,
            },
            "_raw_output": decoded,
            "_parse_error": True,
        }
    except Exception:
        pred = {
            "engine": "qwen-engine-b-sft",
            "source_file": source_file,
            "extracted_fields": {
                "company": None,
                "date": None,
                "address": None,
                "total": None,
            },
            "_raw_output": decoded,
            "_parse_error": True,
        }

    return pred, decoded

In [26]:
import re

def normalize_text(x):
    if x is None:
        return None
    x = str(x).strip().lower()
    x = x.replace("\n", " ")
    x = re.sub(r"\s+", " ", x)
    return x

def normalize_total(x):
    if x is None:
        return None
    x = str(x).strip().lower()
    x = x.replace("rm", "").replace("$", "").replace(",", "").strip()
    return x

def normalize_date(x):
    if x is None:
        return None
    x = str(x).strip().lower()
    # remove time if present
    x = re.sub(r"\s+\d{1,2}:\d{2}(:\d{2})?\s*(am|pm)?", "", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

def values_match(field, pred, truth):
    if field == "total":
        return normalize_total(pred) == normalize_total(truth)
    elif field == "date":
        return normalize_date(pred) == normalize_date(truth)
    else:
        return normalize_text(pred) == normalize_text(truth)

In [27]:
doc_id = test_common[0]

ocr_path = TEST_OCR_DIR / f"{doc_id}.txt"
gt_path = TEST_GT_DIR / f"{doc_id}.txt"

with open(ocr_path, "r", encoding="utf-8") as f:
    ocr_text = f.read()

with open(gt_path, "r", encoding="utf-8") as f:
    gt = json.load(f)

pred, raw_output = generate_prediction(ocr_text, f"{doc_id}.txt")

print("GROUND TRUTH:")
print(gt)

print("\nMODEL PREDICTION:")
print(pred)

print("\nRAW MODEL OUTPUT:")
print(raw_output)

GROUND TRUTH:
{'company': 'OJC MARKETING SDN BHD', 'date': '15/01/2019', 'address': 'NO 2 & 4, JALAN BAYU 4, BANDAR SERI ALAM, B1750 MASAI, JOHOR', 'total': '193.00'}

MODEL PREDICTION:
{'engine': 'qwen-engine-b-sft', 'source_file': 'X51005716394.txt', 'extracted_fields': {'company': 'OJC MARKETING SDN BHD', 'date': '15/01/2019', 'address': 'NO 2 & 4, JALAN BAYU 4, BANDAR SERI ALAM, 81750 MASAI, JOHOR', 'total': '193.00'}}

RAW MODEL OUTPUT:
{
  "engine": "qwen-engine-b-sft",
  "source_file": "X51005716394.txt",
  "extracted_fields": {
    "company": "OJC MARKETING SDN BHD",
    "date": "15/01/2019",
    "address": "NO 2 & 4, JALAN BAYU 4, BANDAR SERI ALAM, 81750 MASAI, JOHOR",
    "total": "193.00"
  }
}Human: Extract the company, date, address, and total from the following OCR receipt text.

OCR TEXT:
 

 

 

 

 

 

 

 

 

 




In [28]:
import re

def normalize(x):
    if x is None:
        return None
    x = str(x).strip().lower()
    x = x.replace("\n", " ")
    x = re.sub(r"\s+", " ", x)   # collapse multiple spaces
    return x

In [29]:
from tqdm.auto import tqdm
import json
import os

PARTIAL_PATH = "../engine_b_qwen/outputs_engine_b/qwen_sft_eval_partial.json"
FULL_PATH = "../engine_b_qwen/outputs_engine_b/qwen_sft_eval_full.json"

FIELDS = ["company", "date", "address", "total"]

os.makedirs("../engine_b_qwen/outputs_engine_b", exist_ok=True)

# Resume safely
if os.path.exists(PARTIAL_PATH):
    with open(PARTIAL_PATH, "r") as f:
        results = json.load(f)
    start_index = len(results)
    print(f"Resuming from {start_index}")
else:
    results = []
    start_index = 0
    print("Starting fresh")

# These only track the current run chunk.
# Final metrics should be recomputed from the full results list afterward.
field_correct = {f: 0 for f in FIELDS}
field_total = {f: 0 for f in FIELDS}

for i, doc_id in enumerate(tqdm(test_common[start_index:], desc="Evaluating"), start=start_index + 1):
    ocr_path = TEST_OCR_DIR / f"{doc_id}.txt"
    gt_path = TEST_GT_DIR / f"{doc_id}.txt"

    with open(ocr_path, "r", encoding="utf-8") as f:
        ocr_text = f.read()

    with open(gt_path, "r", encoding="utf-8") as f:
        gt = json.load(f)

    pred, raw_output = generate_prediction(ocr_text, f"{doc_id}.txt")
    pred_fields = pred.get("extracted_fields", {})

    row = {"doc_id": doc_id}
    for field in FIELDS:
        pred_val = pred_fields.get(field)
        gt_val = gt.get(field)

        p = normalize(pred_val)
        g = normalize(gt_val)
        ok = (p == g)

        row[field] = "✔" if ok else "✘"
        row[f"{field}_pred"] = pred_val
        row[f"{field}_gt"] = gt_val

        field_total[field] += 1
        if ok:
            field_correct[field] += 1

    results.append(row)

    # checkpoint every 5 docs
    if i % 5 == 0:
        with open(PARTIAL_PATH, "w") as f:
            json.dump(results, f, indent=2)

# final save no matter where the loop ended
with open(PARTIAL_PATH, "w") as f:
    json.dump(results, f, indent=2)

with open(FULL_PATH, "w") as f:
    json.dump(results, f, indent=2)

print(f"Saved {len(results)} total rows")

Resuming from 85


Evaluating: 100%|██████████| 262/262 [31:11<00:00,  7.14s/it]

Saved 347 total rows


In [30]:
with open(FULL_PATH, "r") as f:
    saved_results = json.load(f)

field_correct = {f: 0 for f in FIELDS}
field_total = {f: 0 for f in FIELDS}

for row in saved_results:
    for field in FIELDS:
        field_total[field] += 1
        if row.get(field) == "✔":
            field_correct[field] += 1

total_correct = sum(field_correct.values())
total_fields = sum(field_total.values())

print("Field-level accuracy:")
for field in FIELDS:
    c = field_correct[field]
    t = field_total[field]
    acc = 100 * c / t if t else 0
    print(f"{field}: {c}/{t} = {acc:.1f}%")

overall = 100 * total_correct / total_fields if total_fields else 0
print(f"\nOverall accuracy: {total_correct}/{total_fields} = {overall:.1f}%")
print(f"Documents evaluated: {len(saved_results)}")

Field-level accuracy:
company: 248/347 = 71.5%
date: 263/347 = 75.8%
address: 151/347 = 43.5%
total: 247/347 = 71.2%

Overall accuracy: 909/1388 = 65.5%
Documents evaluated: 347


In [32]:
print("===== ENGINE B EVALUATION SUMMARY =====")
print(f"Documents evaluated: {report['num_documents_evaluated']}")
print(f"Field predictions:   {report['num_field_predictions']}")
print(f"Overall field acc:   {report['overall_field_accuracy_percent']}%")
print(f"Document acc:        {report['document_accuracy_percent']}%")
print()

print("Field-level accuracy:")
for field in FIELDS:
    print(
        f" - {field}: "
        f"{report['field_correct_counts'][field]}/{report['field_total_counts'][field]} "
        f"= {report['field_accuracy_percent'][field]}%"
    )

print("\nTop mismatch examples:")
for field in FIELDS:
    print(f"\n[{field}]")
    for item in report["top_mismatches"][field][:3]:
        print(f"  count={item['count']}")
        print(f"    pred: {item['pred']}")
        print(f"    gt  : {item['gt']}")

===== ENGINE B EVALUATION SUMMARY =====
Documents evaluated: 347
Field predictions:   1388
Overall field acc:   65.49%
Document acc:        27.09%

Field-level accuracy:
 - company: 248/347 = 71.47%
 - date: 263/347 = 75.79%
 - address: 151/347 = 43.52%
 - total: 247/347 = 71.18%

Top mismatch examples:

[company]
  count=2
    pred: HOME'S HARMONY @ 1 UTAMA SHOPPING CENTRE
    gt  : MONSIEUR ( M ) SDN. BHD.
  count=2
    pred: TAX INVOICE
    gt  : PACIFIC ASIAN FOOD DELIGHTS S/B
  count=2
    pred: MONSIEUR (M) SDN. BHD.
    gt  : MONSIEUR ( M ) SDN. BHD.

[date]
  count=2
    pred: 02/03/2018
    gt  : 08/05/2016
  count=2
    pred: 28/07/2017
    gt  : 28/07/17
  count=2
    pred: 07-04-18
    gt  : 27-04-18

[address]
  count=4
    pred: 12, JALAN TAMPOI 7/4,KAWASAN PERINDUSTRIAN TAMPOI,81200 JOHOR BAHRU,JOHOR
    gt  : 12, JALAN TAMPOI 7/4,KAWASAN PERINDUSTRIAN TAMPOI,81200 JOHOR BAHRU, JOHOR
  count=4
    pred: LOT P.T. 1111, JALAN KPB 6, KAWASAN PERINDUSTRIAN BALAKONG, 43300 SE

## 14. Generate Report File

In [31]:
import json
import csv
from collections import Counter

FIELDS = ["company", "date", "address", "total"]

FULL_PATH = "../engine_b_qwen/outputs_engine_b/qwen_sft_eval_full.json"
REPORT_JSON = "../engine_b_qwen/outputs_engine_b/qwen_sft_report.json"
REPORT_CSV = "../engine_b_qwen/outputs_engine_b/qwen_sft_report_table.csv"

with open(FULL_PATH, "r") as f:
    results = json.load(f)

# Field-level metrics
field_correct = {f: 0 for f in FIELDS}
field_total = {f: 0 for f in FIELDS}

# Document-level metrics
doc_all_correct = 0

# Error tracking
field_errors = {f: [] for f in FIELDS}
field_error_counts = {f: 0 for f in FIELDS}

for row in results:
    doc_ok = True

    for field in FIELDS:
        ok = (row.get(field) == "✔")
        field_total[field] += 1

        if ok:
            field_correct[field] += 1
        else:
            field_error_counts[field] += 1
            doc_ok = False
            field_errors[field].append({
                "doc_id": row["doc_id"],
                "pred": row.get(f"{field}_pred"),
                "gt": row.get(f"{field}_gt"),
            })

    if doc_ok:
        doc_all_correct += 1

field_accuracy = {
    f: round(100 * field_correct[f] / field_total[f], 2) if field_total[f] else 0.0
    for f in FIELDS
}

total_correct = sum(field_correct.values())
total_fields = sum(field_total.values())
overall_accuracy = round(100 * total_correct / total_fields, 2) if total_fields else 0.0
document_accuracy = round(100 * doc_all_correct / len(results), 2) if results else 0.0

# Most frequent exact mismatches per field
top_mismatches = {}
for field in FIELDS:
    counter = Counter(
        (
            str(err["pred"]).strip() if err["pred"] is not None else "None",
            str(err["gt"]).strip() if err["gt"] is not None else "None",
        )
        for err in field_errors[field]
    )
    top_mismatches[field] = [
        {
            "pred": pred,
            "gt": gt,
            "count": count
        }
        for (pred, gt), count in counter.most_common(10)
    ]

report = {
    "num_documents_evaluated": len(results),
    "num_field_predictions": total_fields,
    "overall_field_accuracy_percent": overall_accuracy,
    "document_accuracy_percent": document_accuracy,
    "field_accuracy_percent": field_accuracy,
    "field_correct_counts": field_correct,
    "field_total_counts": field_total,
    "field_error_counts": field_error_counts,
    "top_mismatches": top_mismatches,
    "sample_errors": {
        field: field_errors[field][:10] for field in FIELDS
    }
}

with open(REPORT_JSON, "w") as f:
    json.dump(report, f, indent=2)

with open(REPORT_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow([
        "doc_id",
        "company_ok", "company_pred", "company_gt",
        "date_ok", "date_pred", "date_gt",
        "address_ok", "address_pred", "address_gt",
        "total_ok", "total_pred", "total_gt",
    ])

    for row in results:
        writer.writerow([
            row["doc_id"],
            row.get("company"), row.get("company_pred"), row.get("company_gt"),
            row.get("date"), row.get("date_pred"), row.get("date_gt"),
            row.get("address"), row.get("address_pred"), row.get("address_gt"),
            row.get("total"), row.get("total_pred"), row.get("total_gt"),
        ])

print("Saved:")
print(" -", REPORT_JSON)
print(" -", REPORT_CSV)

Saved:
 - ../engine_b_qwen/outputs_engine_b/qwen_sft_report.json
 - ../engine_b_qwen/outputs_engine_b/qwen_sft_report_table.csv
